# Project Performance Analysis

This notebook analyzes a synthetic dataset of project metrics, exploring patterns and building predictive models to understand what drives project success.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load dataset
file_path = 'synthetic_projects.csv', df.shape)
df.head()


## Data Overview

Let's look at summary statistics and missing values to understand the dataset.

In [ ]:

# Convert date columns to datetime
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])
# Compute duration
df['duration_days'] = (df['end_date'] - df['start_date']).dt.days

# Summary statistics
summary = df.describe(include='all')
summary


## Exploratory Data Analysis (EDA)

We explore distributions and relationships between variables and project success.

In [ ]:

# Set up plotting style
sns.set(style='whitegrid', palette='Set2')

# Histograms of budgets
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df['planned_budget']/1000, bins=30, color='skyblue', edgecolor='black')
axes[0].set_title('Distribution of Planned Budget (k USD)')
axes[0].set_xlabel('Planned Budget (thousands)')
axes[0].set_ylabel('Count')

axes[1].hist(df['actual_budget']/1000, bins=30, color='salmon', edgecolor='black')
axes[1].set_title('Distribution of Actual Budget (k USD)')
axes[1].set_xlabel('Actual Budget (thousands)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Boxplot of ROI by Success
plt.figure(figsize=(6,4))
sns.boxplot(x='success', y='roi', data=df)
plt.title('ROI by Project Success')
plt.xlabel('Success (1=Yes, 0=No)')
plt.ylabel('ROI')
plt.show()

# Countplot of success across risk levels
plt.figure(figsize=(6,4))
sns.countplot(x='risk_level', hue='success', data=df)
plt.title('Project Success by Risk Level')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.show()


In [ ]:

# Correlation matrix for numeric variables
numeric_cols = ['planned_budget', 'actual_budget', 'team_size', 'manager_experience_years', 'duration_days', 'roi']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix for Numeric Features')
plt.show()


## Predictive Modeling

We'll build a logistic regression and a random forest model to predict project success. Categorical variables will be one-hot encoded.

In [ ]:

# Features and target
X = df.drop(columns=['success', 'start_date', 'end_date'])
y = df['success']

# Identify categorical and numerical columns
categorical_cols = ['priority', 'risk_level', 'complexity']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Preprocess: one-hot encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(drop='first'), categorical_cols)],
    remainder='passthrough'
)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Logistic Regression pipeline
log_reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', LogisticRegression(max_iter=1000))])

log_reg_pipeline.fit(X_train, y_train)
y_pred_log_reg = log_reg_pipeline.predict(X_test)

print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_log_reg))
print('Confusion Matrix:
', confusion_matrix(y_test, y_pred_log_reg))
print('Classification Report:
', classification_report(y_test, y_pred_log_reg))

# Random Forest pipeline
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                            ('model', RandomForestClassifier(n_estimators=200, random_state=42))])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))
print('Confusion Matrix:
', confusion_matrix(y_test, y_pred_rf))
print('Classification Report:
', classification_report(y_test, y_pred_rf))


## Conclusion

In this synthetic dataset, we observed relationships between risk level, complexity, manager experience, and project success. The logistic regression and random forest models both achieved respectable accuracy, indicating that these features can help predict project outcomes. In a real-world setting, further feature engineering and model tuning would be necessary, along with validation on unseen data.

This analysis showcases the ability to explore data, visualize key patterns, and build predictive models — skills that are valuable for business analysts, program managers, and data analysts.